# CineMatch — Movie Recommender System
### Exploratory Data Analysis & Model Walkthrough

This notebook walks through:
1. Dataset exploration and visualization
2. Collaborative Filtering (User-Based & Item-Based)
3. Matrix Factorization via SVD
4. Content-Based Filtering using TF-IDF
5. Hybrid Ensemble and Evaluation


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy.sparse import csr_matrix

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

from src.data_loader import DataLoader
from src.collaborative_filter import CollaborativeFilter
from src.matrix_factorization import MatrixFactorization
from src.content_based import ContentBasedFilter
from src.hybrid_recommender import HybridRecommender
from src.evaluator import Evaluator

print('Imports OK')

## 1. Load & Inspect Data

In [ ]:
loader = DataLoader(data_dir='../data')
loader.load().preprocess()

print('=== Dataset Summary ===')
for k, v in loader.summary().items():
    print(f'  {k}: {v}')

In [ ]:
loader.ratings.head(10)

In [ ]:
loader.movies.head(10)

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CineMatch — Dataset EDA', fontsize=16, fontweight='bold')

# Rating distribution
ax = axes[0, 0]
loader.ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Rating Distribution')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')

# Ratings per user (log scale)
ax = axes[0, 1]
user_counts = loader.ratings['userId'].value_counts()
ax.hist(user_counts, bins=40, color='tomato', edgecolor='white')
ax.set_title('Ratings per User')
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Users')
ax.set_yscale('log')

# Ratings per movie
ax = axes[1, 0]
movie_counts = loader.ratings['movieId'].value_counts()
ax.hist(movie_counts, bins=40, color='mediumseagreen', edgecolor='white')
ax.set_title('Ratings per Movie')
ax.set_xlabel('Number of Ratings')
ax.set_ylabel('Movies')
ax.set_yscale('log')

# Genre frequency
ax = axes[1, 1]
genre_series = loader.movies['genres'].str.split('|').explode()
genre_counts = genre_series.value_counts().head(12)
genre_counts.plot(kind='barh', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('Top Genres in Catalog')
ax.set_xlabel('Movies')

plt.tight_layout()
plt.savefig('../docs/eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. User-Item Matrix & Sparsity

In [ ]:
uim = loader.build_user_item_matrix()
print(f'Matrix shape: {uim.shape}')
print(f'Non-zero entries: {uim.nnz:,}')
print(f'Density: {uim.nnz / (uim.shape[0] * uim.shape[1]):.4%}')

# Visualize a small submatrix
sub = uim[:40, :60].toarray()
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(sub, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_title('User-Item Matrix Heatmap (first 40 users × 60 movies)', fontsize=13)
ax.set_xlabel('Movie Index')
ax.set_ylabel('User Index')
plt.colorbar(im, ax=ax, label='Rating')
plt.tight_layout()
plt.savefig('../docs/matrix_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Matrix Factorization (SVD)

In [ ]:
train, test = loader.train_test_split(test_size=0.2)

# Build train-only matrix
row = train['userId'].map(loader.user_id_map)
col = train['movieId'].map(loader.item_id_map)
train_matrix = csr_matrix((train['rating'].values, (row, col)), shape=uim.shape)

mf = MatrixFactorization(n_factors=50)
mf.fit(train_matrix)
print('MF model trained.')

In [ ]:
# Visualize latent factors for first 100 users using PCA
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
user_2d = pca.fit_transform(mf.user_factors[:100])

fig, ax = plt.subplots(figsize=(9, 6))
scatter = ax.scatter(user_2d[:, 0], user_2d[:, 1], 
                     c=mf.user_means[:100], cmap='viridis', alpha=0.8, s=60)
plt.colorbar(scatter, ax=ax, label='Mean Rating')
ax.set_title('User Latent Factors (PCA 2D) — colored by mean rating', fontsize=12)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
plt.tight_layout()
plt.savefig('../docs/latent_factors.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Sample Recommendations

In [ ]:
# Pick a user with decent activity
active_users = train['userId'].value_counts()
sample_uid = active_users.index[5]

uidx = loader.user_id_map[sample_uid]
mf_recs = mf.recommend(uidx, train_matrix, n=10)

print(f'Top 10 MF recommendations for User {sample_uid}:')
print(f'(User has rated {active_users[sample_uid]} movies)\n')

for rank, (iidx, score) in enumerate(mf_recs, 1):
    mid = loader.reverse_item_map[iidx]
    title = loader.get_movie_title(mid)
    print(f'  {rank:2}. {title:<40} (score: {score:.2f})')

## 6. Hybrid Recommender

In [ ]:
hybrid = HybridRecommender(w_cf=0.3, w_mf=0.5, w_cb=0.2, n_factors=50, cf_k=20)
hybrid.fit(train_matrix, loader.movies, train, loader.user_id_map, loader.item_id_map)
print('Hybrid model trained.')

In [ ]:
recs_df = hybrid.recommend(sample_uid, n=10)
print(f'Hybrid recommendations for User {sample_uid}:')
recs_df

## 7. Evaluation

In [ ]:
ev = Evaluator(train, test, relevance_threshold=4.0)

# MF rating metrics
rating_metrics = ev.evaluate_rating_prediction(
    predict_fn=mf.predict,
    user_id_map=loader.user_id_map,
    item_id_map=loader.item_id_map,
    sample_size=3000,
)
print('Rating Prediction Metrics:')
for k, v in rating_metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Hybrid ranking metrics
def hybrid_rec_fn(user_id, n):
    df = hybrid.recommend(user_id, n=n)
    return df['movieId'].tolist()

rank_metrics = ev.evaluate_ranking(
    recommend_fn=hybrid_rec_fn,
    user_id_map=loader.user_id_map,
    k=10,
    n_users=50,
)
print('Ranking Metrics (Hybrid, K=10):')
for k, v in rank_metrics.items():
    print(f'  {k}: {v}')

## 8. Model Explainability

In [ ]:
# Show per-model score breakdown for a specific (user, movie) pair
sample_movie_id = recs_df.iloc[0]['movieId']
explanation = hybrid.explain(sample_uid, sample_movie_id)

print(f"Score breakdown for User {sample_uid} → '{recs_df.iloc[0]['title']}'")
print(f"  CF score  : {explanation['cf_score']}  (weight: {explanation['weights']['cf']:.0%})")
print(f"  MF score  : {explanation['mf_score']}  (weight: {explanation['weights']['mf']:.0%})")
print(f"  CB score  : {explanation['cb_score']}  (weight: {explanation['weights']['cb']:.0%})")

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
models = ['Collaborative\nFilter', 'Matrix\nFactorization', 'Content\nBased']
scores = [explanation['cf_score'], explanation['mf_score'], explanation['cb_score']]
colors = ['#4C72B0', '#DD8452', '#55A868']
ax.bar(models, scores, color=colors, edgecolor='white', width=0.5)
ax.set_title(f"Model Score Breakdown\nUser {sample_uid} → {recs_df.iloc[0]['title']}", fontsize=12)
ax.set_ylabel('Score')
ax.set_ylim(0, 5)
plt.tight_layout()
plt.savefig('../docs/explainability.png', dpi=150, bbox_inches='tight')
plt.show()